In [10]:
import sys; sys.path.append(r"C:\Users\dillo\OneDrive\Documents\Projects\Microsoft Mind")
import numpy as np, pandas as pd
from src.db import get_engine
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

emb = np.load("../models/news_emb.npy")
news_ids = np.load("../models/news_emb_ids.npy", allow_pickle=True)

engine = get_engine()
meta = (pd.read_sql("SELECT news_id, category, subcategory, title FROM dim_news", engine)
          .set_index("news_id").reindex(news_ids).reset_index())   # row-aligned to emb
print(emb.shape, len(meta))

(51282, 384) 51282


In [11]:
k = 50
km = MiniBatchKMeans(n_clusters=k, random_state=0, n_init=10, batch_size=2048)
labels = km.fit_predict(emb)
meta["cluster"] = labels
print(meta["cluster"].value_counts().head())

cluster
39    1750
42    1669
33    1640
32    1553
22    1548
Name: count, dtype: int64


In [12]:
def show_cluster(c, n=6):
    sub = meta[meta.cluster == c]
    top_cats = sub.category.value_counts().head(3).to_dict()
    print(f"Cluster {c} — {len(sub)} articles | {top_cats}")
    for t in sub.title.head(n):
        print("   ", t)

for c in range(6):
    show_cluster(c)

Cluster 0 — 894 articles | {'news': 628, 'lifestyle': 70, 'video': 54}
    Mom with schizophrenia, 6-year-old daughter missing in Queens
    Child falls from 9th floor balcony in White Oak, Maryland, officials say
    Tour the abandoned Outlaw House with an FBI connection
    The migrant caravans: She misses home and prays not to be sent back
    After months on the loose, an elusive emu has finally been spotted in North Carolina
    Runaway 14-Year-Old Found In Cornfield Near Pennock
Cluster 1 — 1406 articles | {'sports': 1233, 'news': 46, 'health': 30}
    PGA Tour winners
    Randall's Rant: Is Tiger already the GOAT without 18 majors?
    All of Kevin Harvick's Monster Energy Series victories
    Dissecting the softer, gentler Tiger Woods
    American teenager Gauff reaches maiden WTA final in Linz
    Bare Knuckle Knockout! Watch Gabriel Gonzaga Whup Bigfoot Silva In Tampa
Cluster 2 — 1016 articles | {'weather': 760, 'news': 96, 'travel': 40}
    It's been Orlando's hottest Octobe

In [13]:
nn = NearestNeighbors(n_neighbors=2, metric="cosine").fit(emb)
dist, idx = nn.kneighbors(emb)
sim      = 1 - dist[:, 1]
neighbor = idx[:, 1]

THRESH = 0.90
dupe = sim > THRESH
print(f"{dupe.sum()} articles ({dupe.mean():.1%}) have a near-duplicate above {THRESH}")

3017 articles (5.9%) have a near-duplicate above 0.9


In [14]:
pairs = sorted(((sim[i], i, neighbor[i]) for i in np.where(dupe)[0]), reverse=True)
seen = set()
for s, i, j in pairs:
    if j in seen: continue            # avoid printing both directions of a pair
    seen.add(i)
    print(f"{s:.3f}")
    print("   A:", meta.title.iloc[i])
    print("   B:", meta.title.iloc[j])

1.000
   A: Still teaching at 95, Jimmy Carter draws devotees to church
   B: Still teaching at 95, Jimmy Carter draws devotees to church
1.000
   A: Sunday Real Estate: 3 Luxurious Florida Homes
   B: Sunday Real Estate: 3 Luxurious Florida Homes
1.000
   A: Watch: Panthers DE Gerald McCoy called for suspect roughing the passer on Aaron Rodgers
   B: Watch: Panthers DE Gerald McCoy called for suspect roughing the passer on Aaron Rodgers
1.000
   A: Sam Darnold thinks 2-7 Jets still have shot at making playoffs
   B: Sam Darnold thinks 2-7 Jets still have shot at making playoffs
1.000
   A: Airman Missing In Gulf Of Mexico Id'ed As Dallas Native
   B: Airman Missing In Gulf Of Mexico Id'ed As Dallas Native
1.000
   A: Watch: Ravens QB Lamar Jackson busts out spin move on incredible TD run
   B: Watch: Ravens QB Lamar Jackson busts out spin move on incredible TD run
1.000
   A: Watch: Ravens QB Lamar Jackson busts out spin move on incredible TD run
   B: Watch: Ravens QB Lamar Jackson b

In [15]:
row_of = {nid: i for i, nid in enumerate(news_ids)}

def intra_list_diversity(news_id_list, emb, row_of):
    idx = [row_of[n] for n in news_id_list if n in row_of]
    if len(idx) < 2:
        return np.nan
    V = emb[idx]                                
    S = V @ V.T                                   
    n = len(idx)
    off_diag_mean = (S.sum() - n) / (n * (n - 1))   
    return 1 - off_diag_mean

In [16]:
pop = pd.read_sql("""SELECT news_id FROM fact_impressions
                     GROUP BY news_id ORDER BY SUM(clicked) DESC LIMIT 10""", engine)
pop_list = pop["news_id"].tolist()

varied_list = meta.groupby("cluster")["news_id"].first().head(10).tolist()

redundant_list = meta[meta.cluster == 1]["news_id"].head(10).tolist()

In [17]:
for name, lst in [("redundant (1 cluster)", redundant_list),
                  ("popularity (top clicked)", pop_list),
                  ("varied (10 clusters)", varied_list)]:
    print(f"{name:28s} diversity = {intra_list_diversity(lst, emb, row_of):.3f}")

redundant (1 cluster)        diversity = 0.756
popularity (top clicked)     diversity = 0.861
varied (10 clusters)         diversity = 0.955
